# Part II. Inferential Statistics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
TARGET_COL = 'Attrition'
sns.set_style("whitegrid")
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.titlesize': 13})

import os
os.makedirs("results", exist_ok=True)

In [ ]:
df_raw = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df = df_raw.copy()

cols_to_drop = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df[TARGET_COL] = df[TARGET_COL].map({'Yes': 1, 'No': 0})
df = df.drop_duplicates(keep='first').reset_index(drop=True)

num_cols = df.select_dtypes(include=np.number).columns.drop(TARGET_COL).tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
attrition_rate = df[TARGET_COL].mean() * 100

print(f"Dataset: {df.shape[0]:,} rows x {df.shape[1]} cols | Attrition rate: {attrition_rate:.2f}%")

## 2.3 Inferential statistics

### 2.3.1 Normality tests

In [49]:
norm_results = []
for col in num_cols:
    sample = df[col].sample(min(len(df), 5000), random_state=RANDOM_STATE)
    stat, p = stats.shapiro(sample)
    norm_results.append({
        'feature':   col,
        'statistic': round(stat, 4),
        'p_value':   round(p, 4),
        'is_normal': p > 0.05
    })

norm_df = pd.DataFrame(norm_results)
normal_cols     = norm_df[norm_df['is_normal']]['feature'].tolist()
non_normal_cols = norm_df[~norm_df['is_normal']]['feature'].tolist()
norm_df

,feature,statistic,p_value,is_normal
0,Age,0.9774,0.0,False
1,DailyRate,0.9544,0.0,False
2,DistanceFromHome,0.8616,0.0,False
3,Education,0.8954,0.0,False
4,EnvironmentSatisfaction,0.8490,0.0,False
5,HourlyRate,0.9550,0.0,False
6,JobInvolvement,0.8094,0.0,False
7,JobLevel,0.8220,0.0,False
8,JobSatisfaction,0.8456,0.0,False
9,MonthlyIncome,0.8279,0.0,False


### 2.3.2 Group difference tests: Numerical Features

In [50]:
group_test_results = []
stayed_df = df[df[TARGET_COL] == 0]
left_df   = df[df[TARGET_COL] == 1]

for col in num_cols:
    g0, g1 = stayed_df[col], left_df[col]

    if col in normal_cols:
        _, lev_p = stats.levene(g0, g1)
        equal_var = lev_p > 0.05
        stat, p = stats.ttest_ind(g0, g1, equal_var=equal_var)
        test_name = "Welch t-test" if not equal_var else "t-test"
    else:
        stat, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
        test_name = "Mann-Whitney U"

    pooled_std = np.sqrt((g0.std()**2 + g1.std()**2)/2)
    cohen_d = abs(g0.mean() - g1.mean())/pooled_std if pooled_std > 0 else 0

    if cohen_d < 0.2:
        effect = 'Negligible'
    elif cohen_d < 0.5:
        effect = 'Small'
    elif cohen_d < 0.8:
        effect = 'Medium'
    else:
        effect = 'Large'

    group_test_results.append({
        'feature':     col,
        'test':        test_name,
        'mean_stayed': round(g0.mean(), 3),
        'mean_left':   round(g1.mean(), 3),
        'p_value':     round(p, 4),
        'significant': p < 0.05,
        'cohen_d':     round(cohen_d, 3),
        'effect':      effect
    })

num_test_df = pd.DataFrame(group_test_results).sort_values('p_value')
num_test_df

,feature,test,mean_stayed,mean_left,p_value,significant,cohen_d,effect
0,Age,Mann-Whitney U,37.561,33.608,0.0000,True,0.425,Small
7,JobLevel,Mann-Whitney U,2.146,1.637,0.0000,True,0.493,Small
6,JobInvolvement,Mann-Whitney U,2.770,2.519,0.0000,True,0.343,Small
9,MonthlyIncome,Mann-Whitney U,6832.740,4787.093,0.0000,True,0.479,Small
15,StockOptionLevel,Mann-Whitney U,0.845,0.527,0.0000,True,0.374,Small
22,YearsWithCurrManager,Mann-Whitney U,4.367,2.852,0.0000,True,0.449,Small
19,YearsAtCompany,Mann-Whitney U,7.369,5.131,0.0000,True,0.372,Small
16,TotalWorkingYears,Mann-Whitney U,11.863,8.245,0.0000,True,0.484,Small
20,YearsInCurrentRole,Mann-Whitney U,4.484,2.903,0.0000,True,0.462,Small
8,JobSatisfaction,Mann-Whitney U,2.779,2.468,0.0001,True,0.281,Small


In [51]:
num_test_df.to_csv("results/inferential_numerical.csv", index=False)

### 2.3.3 Independence tests: Categorical features

In [52]:
chi2_results = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df[TARGET_COL])
    chi2_stat, p, dof, _ = stats.chi2_contingency(ct)

    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    cramers_v = np.sqrt(chi2_stat/(n*min_dim)) if min_dim>0 else 0

    if cramers_v < 0.1:
        effect = 'Negligible'
    elif cramers_v < 0.3:
        effect = 'Small'
    elif cramers_v < 0.5:
        effect = 'Medium'
    else:
        effect = 'Large'

    chi2_results.append({
        'feature':     col,
        'chi2':        round(chi2_stat, 2),
        'dof':         dof,
        'p_value':     round(p, 4),
        'significant': p < 0.05,
        'cramers_v':   round(cramers_v, 3),
        'effect':      effect
    })

chi2_df = pd.DataFrame(chi2_results).sort_values('p_value')
chi2_df

,feature,chi2,dof,p_value,significant,cramers_v,effect
0,BusinessTravel,24.18,2,0.0000,True,0.128,Small
6,OverTime,87.56,1,0.0000,True,0.244,Small
5,MaritalStatus,46.16,2,0.0000,True,0.177,Small
4,JobRole,86.19,8,0.0000,True,0.242,Small
1,Department,10.80,2,0.0045,True,0.086,Negligible
2,EducationField,16.02,5,0.0068,True,0.104,Small
3,Gender,1.12,1,0.2906,False,0.028,Negligible


In [53]:
chi2_df.to_csv("results/inferential_categorical.csv", index=False)

### 2.3.4 Inferential summary

In [54]:
sig_num_features = num_test_df[
    (num_test_df['significant']) & (num_test_df['cohen_d'] >= 0.2)
]['feature'].tolist()

sig_cat_features = chi2_df[
    (chi2_df['significant']) & (chi2_df['cramers_v'] >= 0.1)
]['feature'].tolist()

not_sig_features = (
    num_test_df[~num_test_df['significant']]['feature'].tolist() +
    chi2_df[~chi2_df['significant']]['feature'].tolist()
)

summary_inferential = pd.DataFrame({
    'category': ['Significant Numerical (effect >= Small)',
                 'Significant Categorical (effect >= Small)',
                 'Not Significant'],
    'count':    [len(sig_num_features), len(sig_cat_features), len(not_sig_features)],
    'features': [', '.join(sig_num_features),
                 ', '.join(sig_cat_features),
                 ', '.join(not_sig_features)]
})
summary_inferential

,category,count,features
0,Significant Numerical (effect >= Small),12,"Age, JobLevel, JobInvolvement, MonthlyIncome, ..."
1,Significant Categorical (effect >= Small),5,"BusinessTravel, OverTime, MaritalStatus, JobRo..."
2,Not Significant,8,"RelationshipSatisfaction, NumCompaniesWorked, ..."
